<a href="https://colab.research.google.com/github/MWANIKID/Signal-or-Redundancy-The-Incremental-Value-of-Technical-Indicators-/blob/main/Signal_or_Redundancy%3F_The_Incremental_Value_of_Technical_Indicators_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# =============================================================================
# SIGNAL OR REDUNDANCY? -- COMPLETE COLAB ML/DL PIPELINE
# =============================================================================
# Inputs:
#   The completed R ZIP:
#   Signal_or_Redundancy_R_FULL_LOGHAR_FIXED_20160801_20260731.zip
#
# Models:
#   1) LightGBM (tabular nonlinear ML)
#   2) Compact pooled LSTM with stock embedding (sequential DL)
#
# Central experiment:
#   A = volatility memory
#   B = A + primitive price/range/activity/liquidity/size information
#   C = B + technical indicators
#
# Horizons:
#   5 days (primary), 10 and 20 days (robustness)
#
# Design:
#   - Hyperparameters are tuned ONLY on B using annual 2020-2023 validation folds.
#   - The chosen B hyperparameters are frozen and reused for A, C, and family
#     ablations. This prevents C from receiving an extra model-search advantage.
#   - Locked test is 2024-01-02 through 2026-07-31.
#   - ML/DL models are re-estimated annually in the locked test using only
#     outcomes whose target windows have fully ended before each refit date.
#   - Technical indicators and targets are NEVER reconstructed in Python.
#   - QLIKE is primary; MSE secondary.
#   - B-vs-C inference uses daily cross-sectional loss differences, HAC and
#     moving-block bootstrap, with Holm correction across horizons.
# =============================================================================

import os, sys, glob, json, math, time, shutil, zipfile, platform, gc, random, warnings, subprocess, importlib.util
from pathlib import Path

# ------------------------------- INSTALL --------------------------------------
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_package("lightgbm", "lightgbm")
ensure_package("tensorflow", "tensorflow")
ensure_package("sklearn", "scikit-learn")
ensure_package("scipy", "scipy")

import numpy as np
import pandas as pd
import lightgbm as lgb
import tensorflow as tf
from scipy.stats import norm
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------- REPRODUCIBILITY ---------------------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

RUN_STARTED_AT = pd.Timestamp.now()

# ------------------------------- CONFIG ---------------------------------------
HORIZONS = [5, 10, 20]
PRIMARY_H = 5
VAR_SCALE = 10000.0
LOG_EPS_SCALED = 1e-8
FORECAST_EPS = 1e-12

DEV_START = pd.Timestamp("2016-08-01")
DEV_END = pd.Timestamp("2023-12-29")
TEST_START = pd.Timestamp("2024-01-02")
TEST_END = pd.Timestamp("2026-07-31")

# Colab test re-estimation is annual. This matches the development-fold
# granularity and keeps ML/DL refitting transparent and computationally stable.
TEST_REFIT_FREQUENCY = "annual"

# LSTM sequence length: one trading month of observed stock records.
SEQ_LEN = 20
LSTM_MAX_EPOCHS = 50
LSTM_PATIENCE = 6

BOOTSTRAP_REPS = 5000
RUN_LGB_FAMILY_ABLATIONS = True
RUN_LSTM_FAMILY_ABLATIONS = True

# Set to a full path if the ZIP is already in Google Drive or /content.
# Leave as None for automatic detection/upload.
R_ZIP_PATH = None

OUT_DIR = Path("/content/Signal_or_Redundancy_COLAB_RESULTS")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = Path("/content/Signal_or_Redundancy_R_EXTRACTED")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_ZIP = Path("/content/Signal_or_Redundancy_COLAB_ML_DL_RESULTS.zip")

# ------------------------------ FEATURES --------------------------------------
A_VARS = [
    "yz_hist_5", "yz_hist_10", "yz_hist_20"
]

B_EXTRA = [
    "ret_cc", "abs_ret_cc", "ret_on", "log_hl_range",
    "log_volume", "log_turnover20", "zero_ret20",
    "log_amihud20", "log_mcap"
]

TREND_VARS = ["sma_gap20", "nmacd_12_26", "adx14"]
MOMENTUM_VARS = ["rsi14", "roc10", "stoch_k14"]
RANGE_TI_VARS = ["natr14", "bbw20"]
TECH_VARS = TREND_VARS + MOMENTUM_VARS + RANGE_TI_VARS

B_VARS = A_VARS + B_EXTRA
C_VARS = B_VARS + TECH_VARS

FEATURE_SPECS = {
    "A_volatility_memory": A_VARS,
    "B_primitive": B_VARS,
    "C_primitive_plus_TI": C_VARS,
    "B_plus_trend": B_VARS + TREND_VARS,
    "B_plus_momentum": B_VARS + MOMENTUM_VARS,
    "B_plus_range_TI": B_VARS + RANGE_TI_VARS,
}

MAIN_SPECS = ["A_volatility_memory", "B_primitive", "C_primitive_plus_TI"]
FAMILY_SPECS = ["B_plus_trend", "B_plus_momentum", "B_plus_range_TI"]

# ----------------------------- INPUT ZIP ---------------------------------------
def locate_r_zip():
    if R_ZIP_PATH is not None:
        p = Path(R_ZIP_PATH)
        if not p.exists():
            raise FileNotFoundError(f"R_ZIP_PATH does not exist: {p}")
        return p

    preferred = sorted(glob.glob(
        "/content/Signal_or_Redundancy_R_FULL_LOGHAR_FIXED_*.zip"
    ))
    if preferred:
        return Path(preferred[-1])

    other = sorted(glob.glob("/content/*.zip"))
    if len(other) == 1:
        return Path(other[0])

    try:
        from google.colab import files
        print("Upload the completed R ZIP file.")
        uploaded = files.upload()
        zips = [Path("/content") / k for k in uploaded if k.lower().endswith(".zip")]
        if not zips:
            raise FileNotFoundError("No ZIP file was uploaded.")
        return zips[0]
    except ImportError as e:
        raise FileNotFoundError(
            "No R ZIP found. Set R_ZIP_PATH to the completed R archive."
        ) from e

R_ZIP = locate_r_zip()
print("Using R ZIP:", R_ZIP)

with zipfile.ZipFile(R_ZIP, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def find_one(filename):
    hits = list(EXTRACT_DIR.rglob(filename))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {len(hits)}"
        )
    return hits[0]

PANEL_FILE = find_one("COLAB_FROZEN_PANEL.csv.gz")
FOLD_FILE = find_one("fold_registry.csv")
R_INCREMENTAL_FILE = find_one("log_har_incremental_B_vs_C.csv")
R_INFERENCE_FILE = find_one("log_har_B_vs_C_inference.csv")
R_METRICS_FILE = find_one("r_model_metrics.csv")
R_FEATURE_MANIFEST_FILE = find_one("feature_manifest.csv")

# ------------------------------- LOAD -----------------------------------------
panel = pd.read_csv(PANEL_FILE, compression="gzip")
panel["Date"] = pd.to_datetime(panel["Date"])

for h in HORIZONS:
    panel[f"target_end_{h}"] = pd.to_datetime(panel[f"target_end_{h}"])

panel = panel.sort_values(["Stock", "Date"]).reset_index(drop=True)
panel["row_id"] = np.arange(len(panel), dtype=np.int64)

stock_levels = sorted(panel["Stock"].astype(str).unique())
stock_to_id = {s: i for i, s in enumerate(stock_levels)}
panel["stock_id"] = panel["Stock"].map(stock_to_id).astype(np.int32)
N_STOCKS = len(stock_levels)

folds = pd.read_csv(FOLD_FILE)
for c in ["train_start", "train_end", "validation_start", "validation_end"]:
    folds[c] = pd.to_datetime(folds[c])

# Validate frozen columns.
required_columns = {
    "Stock", "Date", "Category", "segment_id", "liq_state", "stock_id",
    *A_VARS, *B_EXTRA, *TECH_VARS,
    *[f"target_yz_{h}" for h in HORIZONS],
    *[f"target_end_{h}" for h in HORIZONS],
}
missing_cols = sorted(required_columns - set(panel.columns))
if missing_cols:
    raise ValueError(f"Frozen panel is missing required columns: {missing_cols}")

print("Frozen panel rows:", len(panel))
print("Stocks:", N_STOCKS)
print("Date range:", panel["Date"].min().date(), "to", panel["Date"].max().date())

# -------------------------- LOCKED TEST PERIODS -------------------------------
TEST_PERIODS = pd.DataFrame([
    {
        "label": "test_2024",
        "start": pd.Timestamp("2024-01-02"),
        "end": pd.Timestamp("2024-12-31"),
    },
    {
        "label": "test_2025",
        "start": pd.Timestamp("2025-01-01"),
        "end": pd.Timestamp("2025-12-31"),
    },
    {
        "label": "test_2026",
        "start": pd.Timestamp("2026-01-01"),
        "end": pd.Timestamp("2026-07-31"),
    },
])

# ---------------------------- GENERAL HELPERS ---------------------------------
def target_col(h):
    return f"target_yz_{h}"

def target_end_col(h):
    return f"target_end_{h}"

def target_to_log(y):
    y = np.asarray(y, dtype=np.float64)
    return np.log(np.maximum(y * VAR_SCALE, 0.0) + LOG_EPS_SCALED)

def training_smearing(residual_log, stock_ids, min_stock_n=100):
    """Training-only Duan smearing: stock-specific with global fallback."""
    residual_log = np.asarray(residual_log, dtype=np.float64).reshape(-1)
    stock_ids = np.asarray(stock_ids, dtype=np.int32).reshape(-1)

    exp_resid = np.exp(np.clip(residual_log, -50.0, 50.0))
    good = np.isfinite(exp_resid)
    global_smear = float(np.mean(exp_resid[good])) if np.any(good) else 1.0
    if (not np.isfinite(global_smear)) or global_smear <= 0:
        global_smear = 1.0

    smear_map = {}
    for sid in np.unique(stock_ids):
        vals = exp_resid[(stock_ids == sid) & good]
        if len(vals) >= min_stock_n:
            sm = float(np.mean(vals))
            if np.isfinite(sm) and sm > 0:
                smear_map[int(sid)] = sm
    return global_smear, smear_map

def log_to_target(y_log, stock_ids=None, global_smear=1.0, smear_map=None):
    y_log = np.asarray(y_log, dtype=np.float64).reshape(-1)
    y_scaled = np.exp(np.clip(y_log, -50.0, 50.0))

    if stock_ids is not None:
        stock_ids = np.asarray(stock_ids, dtype=np.int32).reshape(-1)
        smear_map = smear_map or {}
        smear = np.array(
            [smear_map.get(int(s), global_smear) for s in stock_ids],
            dtype=np.float64
        )
        y_scaled = y_scaled * smear
    else:
        y_scaled = y_scaled * float(global_smear)

    return np.maximum(y_scaled / VAR_SCALE, FORECAST_EPS)

def qlike_loss(y, f):
    y = np.maximum(np.asarray(y, dtype=np.float64), FORECAST_EPS)
    f = np.maximum(np.asarray(f, dtype=np.float64), FORECAST_EPS)
    ratio = y / f
    return ratio - np.log(ratio) - 1.0

def mse_loss(y, f):
    y = np.asarray(y, dtype=np.float64)
    f = np.asarray(f, dtype=np.float64)
    return (y - f) ** 2

def mask_training(h, refit_start):
    tc = target_col(h)
    te = target_end_col(h)
    return (
        (panel["Date"] >= DEV_START)
        & (panel["Date"] < refit_start)
        & panel[tc].notna()
        & panel[te].notna()
        & (panel[te] < refit_start)
    ).to_numpy()

def mask_period(h, period_start, period_end):
    tc = target_col(h)
    return (
        (panel["Date"] >= period_start)
        & (panel["Date"] <= period_end)
        & panel[tc].notna()
    ).to_numpy()

def current_complete_mask(indices, features):
    x = panel.loc[indices, features].to_numpy(dtype=np.float64)
    return np.isfinite(x).all(axis=1)

def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    ok = np.isfinite(p)
    pv = p[ok]
    if len(pv) == 0:
        return out
    order = np.argsort(pv)
    ordered = pv[order]
    m = len(ordered)
    adj = np.maximum.accumulate((m - np.arange(m)) * ordered)
    adj = np.minimum(adj, 1.0)
    rev = np.empty(m)
    rev[order] = adj
    out[np.where(ok)[0]] = rev
    return out

def nw_mean_test(x, lag):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 5:
        return dict(n=n, mean=np.nan, se_hac=np.nan, t_hac=np.nan, p_hac=np.nan, lag=lag)
    lag = int(max(0, min(lag, n - 1)))
    mu = x.mean()
    xc = x - mu
    gamma0 = np.sum(xc ** 2) / n
    lrv = gamma0
    for j in range(1, lag + 1):
        gj = np.sum(xc[j:] * xc[:-j]) / n
        wj = 1.0 - j / (lag + 1.0)
        lrv += 2.0 * wj * gj
    se = math.sqrt(max(lrv, 0.0) / n)
    t = mu / se if se > 0 else np.nan
    p = 2.0 * norm.sf(abs(t)) if np.isfinite(t) else np.nan
    return dict(n=n, mean=mu, se_hac=se, t_hac=t, p_hac=p, lag=lag)

def mbb_mean_test(x, block_length, B=BOOTSTRAP_REPS, seed=SEED):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 10:
        return dict(
            boot_reps=B, block_length=block_length,
            ci_low=np.nan, ci_high=np.nan, p_boot=np.nan
        )
    L = int(max(2, min(block_length, n)))
    obs_mean = x.mean()
    x0 = x - obs_mean
    rng = np.random.default_rng(seed)

    means_unc = np.empty(B, dtype=float)
    means_null = np.empty(B, dtype=float)

    blocks_needed = math.ceil(n / L)
    offsets = np.arange(L)

    for b in range(B):
        starts = rng.integers(0, n, size=blocks_needed)
        idx = ((starts[:, None] + offsets[None, :]) % n).ravel()[:n]
        means_unc[b] = x[idx].mean()
        means_null[b] = x0[idx].mean()

    ci_low, ci_high = np.quantile(means_unc, [0.025, 0.975])
    p_boot = np.mean(np.abs(means_null) >= abs(obs_mean))
    return dict(
        boot_reps=B, block_length=L,
        ci_low=float(ci_low), ci_high=float(ci_high),
        p_boot=float(p_boot)
    )

# =============================================================================
# LIGHTGBM
# =============================================================================

LGB_CANDIDATES = [
    dict(candidate="LGB1", learning_rate=0.05, num_leaves=15, min_data_in_leaf=100,
         feature_fraction=0.90, bagging_fraction=0.90, bagging_freq=1, lambda_l2=1.0),
    dict(candidate="LGB2", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100,
         feature_fraction=0.90, bagging_fraction=0.90, bagging_freq=1, lambda_l2=1.0),
    dict(candidate="LGB3", learning_rate=0.03, num_leaves=31, min_data_in_leaf=50,
         feature_fraction=0.90, bagging_fraction=0.90, bagging_freq=1, lambda_l2=1.0),
    dict(candidate="LGB4", learning_rate=0.03, num_leaves=63, min_data_in_leaf=100,
         feature_fraction=0.80, bagging_fraction=0.90, bagging_freq=1, lambda_l2=2.0),
    dict(candidate="LGB5", learning_rate=0.05, num_leaves=31, min_data_in_leaf=200,
         feature_fraction=0.75, bagging_fraction=0.90, bagging_freq=1, lambda_l2=2.0),
    dict(candidate="LGB6", learning_rate=0.03, num_leaves=15, min_data_in_leaf=200,
         feature_fraction=1.00, bagging_fraction=1.00, bagging_freq=0, lambda_l2=1.0),
]

LGB_BASE_PARAMS = dict(
    objective="regression",
    metric="None",
    verbosity=-1,
    seed=SEED,
    feature_fraction_seed=SEED,
    bagging_seed=SEED,
    data_random_seed=SEED,
    deterministic=True,
    force_col_wise=True,
    num_threads=-1,
)

def lgb_qlike_log(preds, dataset):
    ylog = dataset.get_label()
    diff = np.clip(ylog - preds, -50.0, 50.0)
    ratio = np.exp(diff)
    loss = np.mean(ratio - diff - 1.0)
    return "qlike", float(loss), False

def prepare_lgb(indices, features, require_complete=True):
    indices = np.asarray(indices, dtype=np.int64)
    if require_complete:
        ok = current_complete_mask(indices, features)
        indices = indices[ok]
    X = panel.loc[indices, features + ["stock_id"]].copy()
    # Explicit categorical stock ID with GLOBAL frozen categories so training,
    # validation, and test matrices always share identical category metadata.
    X["stock_id"] = pd.Categorical(
        X["stock_id"].astype(int),
        categories=list(range(N_STOCKS))
    )
    return indices, X

def fit_lgb_fold(h, features, train_mask, pred_mask, params, rounds=None, early_stop=True):
    tr_idx = np.flatnonzero(train_mask)
    va_idx = np.flatnonzero(pred_mask)
    tr_idx, Xtr = prepare_lgb(tr_idx, features, require_complete=True)
    va_idx, Xva = prepare_lgb(va_idx, features, require_complete=True)

    if len(tr_idx) < 1000 or len(va_idx) < 100:
        raise RuntimeError(
            f"Insufficient LightGBM data: train={len(tr_idx)}, pred={len(va_idx)}"
        )

    ytr = target_to_log(panel.loc[tr_idx, target_col(h)].to_numpy())
    yva = target_to_log(panel.loc[va_idx, target_col(h)].to_numpy())

    dtr = lgb.Dataset(
        Xtr, label=ytr,
        categorical_feature=["stock_id"],
        free_raw_data=False
    )

    full_params = dict(LGB_BASE_PARAMS)
    full_params.update({k: v for k, v in params.items() if k != "candidate"})

    if early_stop:
        dva = lgb.Dataset(
            Xva, label=yva,
            categorical_feature=["stock_id"],
            reference=dtr,
            free_raw_data=False
        )
        model = lgb.train(
            full_params,
            dtr,
            num_boost_round=2000,
            valid_sets=[dva],
            valid_names=["validation"],
            feval=lgb_qlike_log,
            callbacks=[lgb.early_stopping(75, verbose=False)]
        )
        best_iter = int(model.best_iteration)
    else:
        best_iter = int(rounds)
        model = lgb.train(
            full_params,
            dtr,
            num_boost_round=best_iter
        )

    # Training-only Duan smearing corrects log-to-level retransformation bias.
    pred_log_train = model.predict(Xtr, num_iteration=best_iter)
    global_smear, smear_map = training_smearing(
        ytr - pred_log_train,
        panel.loc[tr_idx, "stock_id"].to_numpy(dtype=np.int32)
    )

    pred_log = model.predict(Xva, num_iteration=best_iter)
    pred = log_to_target(
        pred_log,
        stock_ids=panel.loc[va_idx, "stock_id"].to_numpy(dtype=np.int32),
        global_smear=global_smear,
        smear_map=smear_map
    )
    y = panel.loc[va_idx, target_col(h)].to_numpy(dtype=float)

    return {
        "model": model,
        "indices": va_idx,
        "prediction": pred,
        "target": y,
        "qlike": float(np.mean(qlike_loss(y, pred))),
        "mse": float(np.mean(mse_loss(y, pred))),
        "best_iteration": best_iter,
        "n_train": len(tr_idx),
        "n_pred": len(va_idx),
        "global_smear": global_smear,
        "stock_smear_count": len(smear_map),
    }

# ------------------------- TUNE LIGHTGBM ON B ONLY ----------------------------
print("\n================ LIGHTGBM TUNING STARTED ================")
lgb_search_records = []

for h in HORIZONS:
    for _, fold in folds.iterrows():
        vstart = pd.Timestamp(fold["validation_start"])
        vend = pd.Timestamp(fold["validation_end"])
        train_mask = mask_training(h, vstart)
        pred_mask = mask_period(h, vstart, vend)

        for cand in LGB_CANDIDATES:
            res = fit_lgb_fold(
                h=h,
                features=B_VARS,
                train_mask=train_mask,
                pred_mask=pred_mask,
                params=cand,
                early_stop=True
            )
            lgb_search_records.append({
                "horizon": h,
                "fold": fold["fold"],
                "candidate": cand["candidate"],
                "qlike": res["qlike"],
                "mse": res["mse"],
                "best_iteration": res["best_iteration"],
                "n_train": res["n_train"],
                "n_validation": res["n_pred"],
            })
            del res["model"]
            gc.collect()

lgb_search = pd.DataFrame(lgb_search_records)
lgb_search.to_csv(OUT_DIR / "lightgbm_hyperparameter_search.csv", index=False)

lgb_selected = {}
lgb_selection_rows = []

for h in HORIZONS:
    s = (
        lgb_search[lgb_search["horizon"] == h]
        .groupby("candidate", as_index=False)
        .agg(
            mean_validation_QLIKE=("qlike", "mean"),
            median_best_iteration=("best_iteration", "median")
        )
        .sort_values("mean_validation_QLIKE")
    )
    winner = s.iloc[0]
    cand = next(c for c in LGB_CANDIDATES if c["candidate"] == winner["candidate"])
    rounds = int(max(10, round(winner["median_best_iteration"])))
    lgb_selected[h] = {"params": cand, "rounds": rounds}

    lgb_selection_rows.append({
        "horizon": h,
        "candidate": cand["candidate"],
        "mean_validation_QLIKE": winner["mean_validation_QLIKE"],
        "fixed_test_rounds": rounds,
        **{k: v for k, v in cand.items() if k != "candidate"},
    })

pd.DataFrame(lgb_selection_rows).to_csv(
    OUT_DIR / "lightgbm_selected_hyperparameters.csv", index=False
)
print("Selected LightGBM configurations:")
print(pd.DataFrame(lgb_selection_rows)[
    ["horizon", "candidate", "mean_validation_QLIKE", "fixed_test_rounds"]
])

# ----------------------- LOCKED LIGHTGBM FORECASTS ----------------------------
print("\n================ LIGHTGBM LOCKED TEST STARTED ================")
lgb_pred_frames = []
lgb_fit_log = []

lgb_specs_to_run = MAIN_SPECS + (FAMILY_SPECS if RUN_LGB_FAMILY_ABLATIONS else [])

for h in HORIZONS:
    selected = lgb_selected[h]
    for _, period in TEST_PERIODS.iterrows():
        pstart = pd.Timestamp(period["start"])
        pend = pd.Timestamp(period["end"])
        tr_mask = mask_training(h, pstart)
        pr_mask = mask_period(h, pstart, pend)

        for spec in lgb_specs_to_run:
            res = fit_lgb_fold(
                h=h,
                features=FEATURE_SPECS[spec],
                train_mask=tr_mask,
                pred_mask=pr_mask,
                params=selected["params"],
                rounds=selected["rounds"],
                early_stop=False
            )
            idx = res["indices"]
            frame = panel.loc[idx, ["Stock", "Date", "Category", "liq_state"]].copy()
            frame["target"] = res["target"]
            frame["forecast"] = res["prediction"]
            frame["horizon"] = h
            frame["model"] = "LightGBM"
            frame["specification"] = spec
            frame["stage"] = "locked_test"
            frame["refit_label"] = period["label"]
            lgb_pred_frames.append(frame)

            lgb_fit_log.append({
                "model": "LightGBM",
                "horizon": h,
                "specification": spec,
                "refit_label": period["label"],
                "n_train": res["n_train"],
                "n_pred": res["n_pred"],
                "fixed_rounds": selected["rounds"],
                "candidate": selected["params"]["candidate"],
                "global_smear": res["global_smear"],
                "stock_smear_count": res["stock_smear_count"],
                "status": "ok",
            })
            del res["model"]
            gc.collect()

lgb_preds = pd.concat(lgb_pred_frames, ignore_index=True)
lgb_preds["qlike"] = qlike_loss(lgb_preds["target"], lgb_preds["forecast"])
lgb_preds["mse"] = mse_loss(lgb_preds["target"], lgb_preds["forecast"])
lgb_preds.to_csv(
    OUT_DIR / "lightgbm_locked_forecasts.csv.gz",
    index=False, compression="gzip"
)
pd.DataFrame(lgb_fit_log).to_csv(OUT_DIR / "lightgbm_fit_log.csv", index=False)
print("LightGBM locked forecast rows:", len(lgb_preds))

# =============================================================================
# LSTM SEQUENCE INDEX
# =============================================================================
print("\n================ BUILDING LSTM SEQUENCES ================")

seq_index = np.full((len(panel), SEQ_LEN), -1, dtype=np.int32)

# Never let an LSTM sequence cross a corporate-action reset segment.
grouped = panel.groupby(["Stock", "segment_id"], sort=False, dropna=False).indices
for _, idx in grouped.items():
    idx = np.asarray(idx, dtype=np.int64)
    idx = idx[np.argsort(panel.loc[idx, "Date"].to_numpy())]
    if len(idx) < SEQ_LEN:
        continue
    for j in range(SEQ_LEN - 1, len(idx)):
        seq_index[idx[j], :] = idx[j - SEQ_LEN + 1:j + 1]

panel["lstm_sequence_available"] = seq_index[:, 0] >= 0

seq_audit = (
    panel.groupby("Stock", as_index=False)
    .agg(
        rows=("row_id", "size"),
        sequences_available=("lstm_sequence_available", "sum")
    )
)
seq_audit["sequence_rate"] = seq_audit["sequences_available"] / seq_audit["rows"]
seq_audit.to_csv(OUT_DIR / "lstm_sequence_audit.csv", index=False)

# Raw numeric matrices cached by specification.
SPEC_RAW = {
    spec: panel[features].to_numpy(dtype=np.float32)
    for spec, features in FEATURE_SPECS.items()
}

def fit_sequence_scaler(X):
    # X shape = n x seq_len x p. Missing historical values are imputed with
    # TRAINING-ONLY medians; then scaled with TRAINING-ONLY means/SDs.
    flat = X.reshape(-1, X.shape[-1]).astype(np.float64)
    med = np.nanmedian(np.where(np.isfinite(flat), flat, np.nan), axis=0)
    med[~np.isfinite(med)] = 0.0
    filled = np.where(np.isfinite(flat), flat, med)
    mean = filled.mean(axis=0)
    sd = filled.std(axis=0)
    sd[~np.isfinite(sd) | (sd < 1e-8)] = 1.0
    return med.astype(np.float32), mean.astype(np.float32), sd.astype(np.float32)

def apply_sequence_scaler(X, med, mean, sd):
    X = X.astype(np.float32, copy=True)
    shape = X.shape
    flat = X.reshape(-1, shape[-1])
    flat = np.where(np.isfinite(flat), flat, med)
    flat = (flat - mean) / sd
    return flat.reshape(shape).astype(np.float32)

def prepare_lstm_data(h, spec, train_mask, pred_mask):
    features = FEATURE_SPECS[spec]
    raw = SPEC_RAW[spec]

    tr_idx = np.flatnonzero(train_mask & panel["lstm_sequence_available"].to_numpy())
    pr_idx = np.flatnonzero(pred_mask & panel["lstm_sequence_available"].to_numpy())

    # At the forecast origin, every feature in the information set must exist.
    # Missing values are imputed only for EARLIER steps inside an otherwise
    # valid sequence, never for the origin's contemporaneous information.
    tr_current = raw[tr_idx]
    pr_current = raw[pr_idx]
    tr_idx = tr_idx[np.isfinite(tr_current).all(axis=1)]
    pr_idx = pr_idx[np.isfinite(pr_current).all(axis=1)]

    if len(tr_idx) < 1000 or len(pr_idx) < 100:
        raise RuntimeError(
            f"Insufficient LSTM sequence data: train={len(tr_idx)}, pred={len(pr_idx)}"
        )

    Xtr_raw = raw[seq_index[tr_idx]]
    Xpr_raw = raw[seq_index[pr_idx]]

    med, mean, sd = fit_sequence_scaler(Xtr_raw)
    Xtr = apply_sequence_scaler(Xtr_raw, med, mean, sd)
    Xpr = apply_sequence_scaler(Xpr_raw, med, mean, sd)

    ytr = target_to_log(panel.loc[tr_idx, target_col(h)].to_numpy()).astype(np.float32)
    ypr = target_to_log(panel.loc[pr_idx, target_col(h)].to_numpy()).astype(np.float32)

    str_id = panel.loc[tr_idx, "stock_id"].to_numpy(dtype=np.int32).reshape(-1, 1)
    spr_id = panel.loc[pr_idx, "stock_id"].to_numpy(dtype=np.int32).reshape(-1, 1)

    return {
        "train_idx": tr_idx,
        "pred_idx": pr_idx,
        "Xtr": Xtr,
        "Xpr": Xpr,
        "stock_tr": str_id,
        "stock_pr": spr_id,
        "ytr_log": ytr,
        "ypr_log": ypr,
        "target_pr": panel.loc[pr_idx, target_col(h)].to_numpy(dtype=float),
        "scaler_median": med,
        "scaler_mean": mean,
        "scaler_sd": sd,
    }

# =============================================================================
# LSTM MODEL
# =============================================================================

LSTM_CANDIDATES = [
    dict(candidate="LSTM1", lstm_units=32, dense_units=16, dropout=0.10,
         learning_rate=1e-3, batch_size=512, embedding_dim=4),
    dict(candidate="LSTM2", lstm_units=64, dense_units=32, dropout=0.20,
         learning_rate=7e-4, batch_size=512, embedding_dim=4),
    dict(candidate="LSTM3", lstm_units=48, dense_units=24, dropout=0.10,
         learning_rate=5e-4, batch_size=512, embedding_dim=6),
]

def qlike_log_metric(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    d = tf.clip_by_value(y_true - y_pred, -30.0, 30.0)
    return tf.reduce_mean(tf.exp(d) - d - 1.0)

def build_lstm_model(n_features, hp):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    seq_in = tf.keras.Input(
        shape=(SEQ_LEN, n_features), name="sequence", dtype=tf.float32
    )
    stock_in = tf.keras.Input(shape=(1,), name="stock_id", dtype=tf.int32)

    x = tf.keras.layers.LSTM(
        hp["lstm_units"],
        dropout=hp["dropout"],
        recurrent_dropout=0.0,
        name="lstm"
    )(seq_in)

    emb = tf.keras.layers.Embedding(
        input_dim=N_STOCKS,
        output_dim=hp["embedding_dim"],
        name="stock_embedding"
    )(stock_in)
    emb = tf.keras.layers.Flatten()(emb)

    x = tf.keras.layers.Concatenate()([x, emb])
    x = tf.keras.layers.Dense(hp["dense_units"], activation="relu")(x)
    x = tf.keras.layers.Dropout(hp["dropout"])(x)

    # Linear output on log-variance scale. Exponentiation at prediction time
    # guarantees a positive variance forecast.
    out = tf.keras.layers.Dense(1, activation="linear", name="log_variance")(x)

    model = tf.keras.Model(inputs=[seq_in, stock_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp["learning_rate"]),
        loss="mse",
        metrics=[qlike_log_metric]
    )
    return model

def fit_lstm_validation(data, hp):
    model = build_lstm_model(data["Xtr"].shape[-1], hp)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_qlike_log_metric",
            mode="min",
            patience=LSTM_PATIENCE,
            restore_best_weights=True,
            verbose=0
        )
    ]

    hist = model.fit(
        [data["Xtr"], data["stock_tr"]],
        data["ytr_log"],
        validation_data=(
            [data["Xpr"], data["stock_pr"]],
            data["ypr_log"]
        ),
        epochs=LSTM_MAX_EPOCHS,
        batch_size=hp["batch_size"],
        shuffle=True,
        verbose=0,
        callbacks=callbacks
    )

    val_hist = np.asarray(hist.history["val_qlike_log_metric"], dtype=float)
    best_epoch = int(np.nanargmin(val_hist) + 1)

    pred_log_train = model.predict(
        [data["Xtr"], data["stock_tr"]],
        batch_size=hp["batch_size"],
        verbose=0
    ).reshape(-1)
    global_smear, smear_map = training_smearing(
        data["ytr_log"] - pred_log_train,
        data["stock_tr"].reshape(-1)
    )

    pred_log = model.predict(
        [data["Xpr"], data["stock_pr"]],
        batch_size=hp["batch_size"],
        verbose=0
    ).reshape(-1)

    pred = log_to_target(
        pred_log,
        stock_ids=data["stock_pr"].reshape(-1),
        global_smear=global_smear,
        smear_map=smear_map
    )
    y = data["target_pr"]

    return {
        "model": model,
        "prediction": pred,
        "qlike": float(np.mean(qlike_loss(y, pred))),
        "mse": float(np.mean(mse_loss(y, pred))),
        "best_epoch": best_epoch,
        "n_train": len(data["train_idx"]),
        "n_pred": len(data["pred_idx"]),
        "global_smear": global_smear,
        "stock_smear_count": len(smear_map),
    }

def fit_lstm_fixed(data, hp, epochs):
    model = build_lstm_model(data["Xtr"].shape[-1], hp)
    model.fit(
        [data["Xtr"], data["stock_tr"]],
        data["ytr_log"],
        epochs=int(epochs),
        batch_size=hp["batch_size"],
        shuffle=True,
        verbose=0
    )
    pred_log_train = model.predict(
        [data["Xtr"], data["stock_tr"]],
        batch_size=hp["batch_size"],
        verbose=0
    ).reshape(-1)
    global_smear, smear_map = training_smearing(
        data["ytr_log"] - pred_log_train,
        data["stock_tr"].reshape(-1)
    )

    pred_log = model.predict(
        [data["Xpr"], data["stock_pr"]],
        batch_size=hp["batch_size"],
        verbose=0
    ).reshape(-1)
    pred = log_to_target(
        pred_log,
        stock_ids=data["stock_pr"].reshape(-1),
        global_smear=global_smear,
        smear_map=smear_map
    )
    return {
        "model": model,
        "prediction": pred,
        "n_train": len(data["train_idx"]),
        "n_pred": len(data["pred_idx"]),
        "global_smear": global_smear,
        "stock_smear_count": len(smear_map),
    }

# ---------------------------- TUNE LSTM ON B ----------------------------------
print("\n================ LSTM TUNING STARTED ================")
lstm_search_records = []

for h in HORIZONS:
    for _, fold in folds.iterrows():
        vstart = pd.Timestamp(fold["validation_start"])
        vend = pd.Timestamp(fold["validation_end"])

        data = prepare_lstm_data(
            h=h,
            spec="B_primitive",
            train_mask=mask_training(h, vstart),
            pred_mask=mask_period(h, vstart, vend)
        )

        for hp in LSTM_CANDIDATES:
            res = fit_lstm_validation(data, hp)
            lstm_search_records.append({
                "horizon": h,
                "fold": fold["fold"],
                "candidate": hp["candidate"],
                "qlike": res["qlike"],
                "mse": res["mse"],
                "best_epoch": res["best_epoch"],
                "n_train": res["n_train"],
                "n_validation": res["n_pred"],
            })
            del res["model"]
            tf.keras.backend.clear_session()
            gc.collect()

        del data
        gc.collect()

lstm_search = pd.DataFrame(lstm_search_records)
lstm_search.to_csv(OUT_DIR / "lstm_hyperparameter_search.csv", index=False)

lstm_selected = {}
lstm_selection_rows = []

for h in HORIZONS:
    s = (
        lstm_search[lstm_search["horizon"] == h]
        .groupby("candidate", as_index=False)
        .agg(
            mean_validation_QLIKE=("qlike", "mean"),
            median_best_epoch=("best_epoch", "median")
        )
        .sort_values("mean_validation_QLIKE")
    )
    winner = s.iloc[0]
    hp = next(c for c in LSTM_CANDIDATES if c["candidate"] == winner["candidate"])
    epochs = int(max(2, round(winner["median_best_epoch"])))
    lstm_selected[h] = {"hp": hp, "epochs": epochs}

    lstm_selection_rows.append({
        "horizon": h,
        "candidate": hp["candidate"],
        "mean_validation_QLIKE": winner["mean_validation_QLIKE"],
        "fixed_test_epochs": epochs,
        **{k: v for k, v in hp.items() if k != "candidate"},
    })

pd.DataFrame(lstm_selection_rows).to_csv(
    OUT_DIR / "lstm_selected_hyperparameters.csv", index=False
)
print("Selected LSTM configurations:")
print(pd.DataFrame(lstm_selection_rows)[
    ["horizon", "candidate", "mean_validation_QLIKE", "fixed_test_epochs"]
])

# --------------------------- LOCKED LSTM TEST ---------------------------------
print("\n================ LSTM LOCKED TEST STARTED ================")
lstm_pred_frames = []
lstm_fit_log = []

lstm_specs_to_run = MAIN_SPECS + (FAMILY_SPECS if RUN_LSTM_FAMILY_ABLATIONS else [])

for h in HORIZONS:
    selected = lstm_selected[h]

    for _, period in TEST_PERIODS.iterrows():
        pstart = pd.Timestamp(period["start"])
        pend = pd.Timestamp(period["end"])
        tr_mask = mask_training(h, pstart)
        pr_mask = mask_period(h, pstart, pend)

        for spec in lstm_specs_to_run:
            data = prepare_lstm_data(
                h=h,
                spec=spec,
                train_mask=tr_mask,
                pred_mask=pr_mask
            )

            res = fit_lstm_fixed(
                data=data,
                hp=selected["hp"],
                epochs=selected["epochs"]
            )

            idx = data["pred_idx"]
            frame = panel.loc[idx, ["Stock", "Date", "Category", "liq_state"]].copy()
            frame["target"] = data["target_pr"]
            frame["forecast"] = res["prediction"]
            frame["horizon"] = h
            frame["model"] = "LSTM"
            frame["specification"] = spec
            frame["stage"] = "locked_test"
            frame["refit_label"] = period["label"]
            lstm_pred_frames.append(frame)

            lstm_fit_log.append({
                "model": "LSTM",
                "horizon": h,
                "specification": spec,
                "refit_label": period["label"],
                "n_train": res["n_train"],
                "n_pred": res["n_pred"],
                "fixed_epochs": selected["epochs"],
                "candidate": selected["hp"]["candidate"],
                "global_smear": res["global_smear"],
                "stock_smear_count": res["stock_smear_count"],
                "status": "ok",
            })

            del res["model"], res, data
            tf.keras.backend.clear_session()
            gc.collect()

lstm_preds = pd.concat(lstm_pred_frames, ignore_index=True)
lstm_preds["qlike"] = qlike_loss(lstm_preds["target"], lstm_preds["forecast"])
lstm_preds["mse"] = mse_loss(lstm_preds["target"], lstm_preds["forecast"])

lstm_preds.to_csv(
    OUT_DIR / "lstm_locked_forecasts.csv.gz",
    index=False, compression="gzip"
)
pd.DataFrame(lstm_fit_log).to_csv(OUT_DIR / "lstm_fit_log.csv", index=False)
print("LSTM locked forecast rows:", len(lstm_preds))

# =============================================================================
# ML/DL EVALUATION
# =============================================================================
preds = pd.concat([lgb_preds, lstm_preds], ignore_index=True)

# Positivity audit.
positivity = (
    preds.groupby(["model", "specification", "horizon"], as_index=False)
    .agg(
        n_forecasts=("forecast", "size"),
        n_nonpositive=("forecast", lambda x: int(np.sum(np.asarray(x) <= 0))),
        min_forecast=("forecast", "min"),
        p001_forecast=("forecast", lambda x: float(np.quantile(x, 0.001)))
    )
)
positivity.to_csv(OUT_DIR / "ml_dl_positivity_audit.csv", index=False)

# Pooled metrics.
metrics = (
    preds.groupby(["model", "specification", "horizon"], as_index=False)
    .agg(
        n=("forecast", "size"),
        QLIKE=("qlike", "mean"),
        MSE=("mse", "mean")
    )
)
metrics.to_csv(OUT_DIR / "ml_dl_model_metrics.csv", index=False)

# Stock-level and equal-stock metrics.
stock_metrics = (
    preds.groupby(["Stock", "model", "specification", "horizon"], as_index=False)
    .agg(
        n=("forecast", "size"),
        QLIKE=("qlike", "mean"),
        MSE=("mse", "mean")
    )
)
stock_metrics.to_csv(OUT_DIR / "ml_dl_stock_level_metrics.csv", index=False)

equal_stock = (
    stock_metrics.groupby(["model", "specification", "horizon"], as_index=False)
    .agg(
        n_stocks=("Stock", "nunique"),
        mean_stock_QLIKE=("QLIKE", "mean"),
        median_stock_QLIKE=("QLIKE", "median"),
        mean_stock_MSE=("MSE", "mean"),
        median_stock_MSE=("MSE", "median")
    )
)
equal_stock.to_csv(OUT_DIR / "ml_dl_equal_stock_weight_metrics.csv", index=False)

# ---------------------------- B vs C ------------------------------------------
def merge_spec_pair(model_name, spec_left, spec_right):
    left = preds[
        (preds["model"] == model_name) &
        (preds["specification"] == spec_left)
    ][["Stock", "Date", "Category", "liq_state", "horizon", "target", "forecast"]].copy()
    left = left.rename(columns={"forecast": "forecast_left"})

    right = preds[
        (preds["model"] == model_name) &
        (preds["specification"] == spec_right)
    ][["Stock", "Date", "horizon", "target", "forecast"]].copy()
    right = right.rename(columns={"target": "target_right", "forecast": "forecast_right"})

    m = left.merge(right, on=["Stock", "Date", "horizon"], how="inner")
    m = m[
        np.isfinite(m["target"]) &
        np.isfinite(m["target_right"]) &
        np.isfinite(m["forecast_left"]) &
        np.isfinite(m["forecast_right"])
    ].copy()
    return m

incremental_rows = []
liquidity_rows = []
daily_diff_frames = []
stock_gain_frames = []

for model_name in ["LightGBM", "LSTM"]:
    bc = merge_spec_pair(model_name, "B_primitive", "C_primitive_plus_TI")
    bc["qB"] = qlike_loss(bc["target"], bc["forecast_left"])
    bc["qC"] = qlike_loss(bc["target"], bc["forecast_right"])
    bc["mB"] = mse_loss(bc["target"], bc["forecast_left"])
    bc["mC"] = mse_loss(bc["target"], bc["forecast_right"])
    bc["d_qlike"] = bc["qB"] - bc["qC"]
    bc["d_mse"] = bc["mB"] - bc["mC"]

    for h, g in bc.groupby("horizon"):
        incremental_rows.append({
            "model": model_name,
            "stage": "locked_test",
            "horizon": int(h),
            "n": len(g),
            "QLIKE_B": g["qB"].mean(),
            "QLIKE_C": g["qC"].mean(),
            "Incremental_Gain_QLIKE_pct":
                100.0 * (g["qB"].mean() - g["qC"].mean()) / g["qB"].mean(),
            "MSE_B": g["mB"].mean(),
            "MSE_C": g["mC"].mean(),
            "Incremental_Gain_MSE_pct":
                100.0 * (g["mB"].mean() - g["mC"].mean()) / g["mB"].mean(),
        })

    for (h, liq), g in bc.dropna(subset=["liq_state"]).groupby(["horizon", "liq_state"]):
        liquidity_rows.append({
            "model": model_name,
            "horizon": int(h),
            "liq_state": liq,
            "n": len(g),
            "QLIKE_B": g["qB"].mean(),
            "QLIKE_C": g["qC"].mean(),
            "Incremental_Gain_QLIKE_pct":
                100.0 * (g["qB"].mean() - g["qC"].mean()) / g["qB"].mean(),
            "MSE_B": g["mB"].mean(),
            "MSE_C": g["mC"].mean(),
            "Incremental_Gain_MSE_pct":
                100.0 * (g["mB"].mean() - g["mC"].mean()) / g["mB"].mean(),
        })

    daily = (
        bc.groupby(["horizon", "Date"], as_index=False)
        .agg(
            n_stocks=("Stock", "size"),
            d_qlike=("d_qlike", "mean"),
            d_mse=("d_mse", "mean")
        )
    )
    daily["model"] = model_name
    daily_diff_frames.append(daily)

    stockg = (
        bc.groupby(["horizon", "Stock"], as_index=False)
        .agg(qB=("qB", "mean"), qC=("qC", "mean"), mB=("mB", "mean"), mC=("mC", "mean"))
    )
    stockg["Incremental_Gain_QLIKE_pct"] = 100.0 * (stockg["qB"] - stockg["qC"]) / stockg["qB"]
    stockg["Incremental_Gain_MSE_pct"] = 100.0 * (stockg["mB"] - stockg["mC"]) / stockg["mB"]
    stockg["model"] = model_name
    stock_gain_frames.append(stockg)

incremental = pd.DataFrame(incremental_rows)
incremental.to_csv(OUT_DIR / "ml_dl_incremental_B_vs_C.csv", index=False)

liquidity = pd.DataFrame(liquidity_rows)
liquidity.to_csv(OUT_DIR / "ml_dl_liquidity_heterogeneity.csv", index=False)

daily_diff = pd.concat(daily_diff_frames, ignore_index=True)
daily_diff.to_csv(OUT_DIR / "ml_dl_daily_loss_differentials.csv", index=False)

stock_gains = pd.concat(stock_gain_frames, ignore_index=True)
stock_gains.to_csv(OUT_DIR / "ml_dl_stock_B_vs_C_gains.csv", index=False)

# ----------------------------- INFERENCE --------------------------------------
inference_rows = []

for model_name in ["LightGBM", "LSTM"]:
    for h in HORIZONS:
        x = (
            daily_diff[
                (daily_diff["model"] == model_name) &
                (daily_diff["horizon"] == h)
            ]
            .sort_values("Date")["d_qlike"]
            .to_numpy(dtype=float)
        )
        hac = nw_mean_test(x, lag=max(h - 1, 1))
        boot = mbb_mean_test(
            x,
            block_length=max(h, 10),
            B=BOOTSTRAP_REPS,
            seed=SEED + h + (100 if model_name == "LSTM" else 0)
        )
        inference_rows.append({
            "model": model_name,
            "stage": "locked_test",
            "horizon": h,
            **hac,
            **boot,
        })

inference = pd.DataFrame(inference_rows)
inference["p_hac_holm"] = np.nan
inference["p_boot_holm"] = np.nan

for model_name in inference["model"].unique():
    ix = inference["model"] == model_name
    inference.loc[ix, "p_hac_holm"] = holm_adjust(inference.loc[ix, "p_hac"].to_numpy())
    inference.loc[ix, "p_boot_holm"] = holm_adjust(inference.loc[ix, "p_boot"].to_numpy())

inference.to_csv(OUT_DIR / "ml_dl_B_vs_C_inference.csv", index=False)

# -------------------------- FAMILY ABLATIONS ----------------------------------
family_rows = []

for model_name in ["LightGBM", "LSTM"]:
    enabled = (
        RUN_LGB_FAMILY_ABLATIONS if model_name == "LightGBM"
        else RUN_LSTM_FAMILY_ABLATIONS
    )
    if not enabled:
        continue

    for family_spec in FAMILY_SPECS:
        fam = merge_spec_pair(model_name, "B_primitive", family_spec)
        fam["qB"] = qlike_loss(fam["target"], fam["forecast_left"])
        fam["qF"] = qlike_loss(fam["target"], fam["forecast_right"])
        fam["mB"] = mse_loss(fam["target"], fam["forecast_left"])
        fam["mF"] = mse_loss(fam["target"], fam["forecast_right"])

        for h, g in fam.groupby("horizon"):
            family_rows.append({
                "model": model_name,
                "family": family_spec,
                "horizon": int(h),
                "n": len(g),
                "QLIKE_B": g["qB"].mean(),
                "QLIKE_Family": g["qF"].mean(),
                "Incremental_Gain_QLIKE_pct":
                    100.0 * (g["qB"].mean() - g["qF"].mean()) / g["qB"].mean(),
                "MSE_B": g["mB"].mean(),
                "MSE_Family": g["mF"].mean(),
                "Incremental_Gain_MSE_pct":
                    100.0 * (g["mB"].mean() - g["mF"].mean()) / g["mB"].mean(),
            })

family_results = pd.DataFrame(family_rows)
family_results.to_csv(OUT_DIR / "ml_dl_indicator_family_ablations.csv", index=False)

# ------------------- INFORMATION-SET PROGRESSION A -> B -> C ------------------
progression_rows = []

for model_name in ["LightGBM", "LSTM"]:
    mm = metrics[metrics["model"] == model_name].copy()
    for h in HORIZONS:
        m = mm[mm["horizon"] == h].set_index("specification")
        if all(s in m.index for s in MAIN_SPECS):
            qa = float(m.loc["A_volatility_memory", "QLIKE"])
            qb = float(m.loc["B_primitive", "QLIKE"])
            qc = float(m.loc["C_primitive_plus_TI", "QLIKE"])
            progression_rows.append({
                "model": model_name,
                "horizon": h,
                "QLIKE_A": qa,
                "QLIKE_B": qb,
                "QLIKE_C": qc,
                "A_to_B_Gain_pct": 100.0 * (qa - qb) / qa,
                "B_to_C_Gain_pct": 100.0 * (qb - qc) / qb,
            })

progression = pd.DataFrame(progression_rows)
progression.to_csv(OUT_DIR / "ml_dl_information_set_progression.csv", index=False)

# =============================================================================
# COMBINE WITH R LOG-HAR RESULTS
# =============================================================================
r_inc = pd.read_csv(R_INCREMENTAL_FILE)
r_inf = pd.read_csv(R_INFERENCE_FILE)

r_inc = r_inc[r_inc["stage"] == "locked_test"].copy()
r_inf = r_inf[r_inf["stage"] == "locked_test"].copy()

r_summary = r_inc.merge(
    r_inf[["horizon", "p_hac_holm", "p_boot_holm"]],
    on="horizon",
    how="left"
)
r_summary["model"] = "LOG_HAR_X"

py_summary = incremental.merge(
    inference[["model", "horizon", "p_hac_holm", "p_boot_holm"]],
    on=["model", "horizon"],
    how="left"
)

summary_cols = [
    "model", "horizon", "n", "QLIKE_B", "QLIKE_C",
    "Incremental_Gain_QLIKE_pct",
    "MSE_B", "MSE_C", "Incremental_Gain_MSE_pct",
    "p_hac_holm", "p_boot_holm"
]

combined_summary = pd.concat(
    [r_summary[summary_cols], py_summary[summary_cols]],
    ignore_index=True
).sort_values(["horizon", "model"])

combined_summary.to_csv(
    OUT_DIR / "FINAL_COMBINED_ARCHITECTURE_SUMMARY.csv", index=False
)

print("\n================ COMBINED LOCKED-TEST SUMMARY ================")
print(combined_summary.to_string(index=False))

# =============================================================================
# REPRODUCIBILITY / README
# =============================================================================
config = {
    "seed": SEED,
    "horizons": HORIZONS,
    "primary_horizon": PRIMARY_H,
    "development_end": str(DEV_END.date()),
    "locked_test_start": str(TEST_START.date()),
    "locked_test_end": str(TEST_END.date()),
    "test_refit_frequency": TEST_REFIT_FREQUENCY,
    "lstm_sequence_length": SEQ_LEN,
    "lstm_max_epochs_tuning": LSTM_MAX_EPOCHS,
    "lstm_patience": LSTM_PATIENCE,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "feature_specs": FEATURE_SPECS,
    "note": (
        "Hyperparameters are tuned on B only and frozen for A/C/family ablations. "
        "Targets and technical indicators come directly from the R frozen panel."
    )
}
with open(OUT_DIR / "colab_run_configuration.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "lightgbm": lgb.__version__,
    "tensorflow": tf.__version__,
}
with open(OUT_DIR / "python_package_versions.json", "w", encoding="utf-8") as f:
    json.dump(versions, f, indent=2)

run_finished = pd.Timestamp.now()
runtime = pd.DataFrame([{
    "run_started_at": str(RUN_STARTED_AT),
    "run_finished_at": str(run_finished),
    "elapsed_minutes": (run_finished - RUN_STARTED_AT).total_seconds() / 60.0,
    "gpu_available": bool(tf.config.list_physical_devices("GPU")),
}])
runtime.to_csv(OUT_DIR / "colab_runtime_manifest.csv", index=False)

readme = f"""SIGNAL OR REDUNDANCY? -- COLAB ML/DL RESULTS

Input R archive:
{R_ZIP.name}

Frozen panel:
{PANEL_FILE.name}

Models:
- LightGBM: pooled nonlinear tabular learner with stock categorical effect.
- LSTM: pooled {SEQ_LEN}-observation sequence model with stock embedding.

Information sets:
A = volatility memory
B = A + primitive price/range/activity/liquidity/size information
C = B + technical indicators

Model tuning:
- LightGBM and LSTM hyperparameters were tuned ONLY on B.
- Annual 2020, 2021, 2022, 2023 chronological validation folds.
- Winning B hyperparameters were frozen and reused for A, C, and family ablations.\n- Log-to-level forecasts use training-only Duan smearing with stock-specific factors when supported.
- No locked-test result was used for model selection.

Locked test:
{TEST_START.date()} to {TEST_END.date()}
Annual recursive re-estimation at the start of 2024, 2025, and 2026.
Training targets were included only when their target window ended before the refit date.

Primary loss:
QLIKE

Secondary loss:
MSE

Primary inference:
Daily cross-sectional mean B-minus-C QLIKE differential;
Newey-West/HAC lag max(h-1,1);
moving-block bootstrap; Holm correction across 5/10/20-day horizons.

IMPORTANT:
Interpret cross-architecture agreement primarily through the sign and statistical
robustness of B-vs-C incremental information. The study is not designed as a
horse race between Log-HAR, LightGBM, and LSTM.
"""
(OUT_DIR / "README_RESULTS.txt").write_text(readme, encoding="utf-8")

# Copy the key R reference tables into the final Colab results for a self-contained archive.
r_ref_dir = OUT_DIR / "R_REFERENCE"
r_ref_dir.mkdir(exist_ok=True)
for src in [
    R_INCREMENTAL_FILE,
    R_INFERENCE_FILE,
    R_METRICS_FILE,
    R_FEATURE_MANIFEST_FILE,
    FOLD_FILE
]:
    shutil.copy2(src, r_ref_dir / src.name)

# ------------------------------ FINAL ZIP -------------------------------------
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

shutil.make_archive(
    str(FINAL_ZIP.with_suffix("")),
    "zip",
    root_dir=OUT_DIR
)

print("\n============================================================")
print("COLAB PIPELINE COMPLETE")
print("============================================================")
print("Results folder:", OUT_DIR)
print("Final ZIP:", FINAL_ZIP)
print("Key file: FINAL_COMBINED_ARCHITECTURE_SUMMARY.csv")
print("============================================================")

# Trigger browser download when running in Google Colab.
try:
    from google.colab import files
    files.download(str(FINAL_ZIP))
except Exception:
    pass


Upload the completed R ZIP file.


Saving Signal_or_Redundancy_R_FULL_LOGHAR_FIXED_20160801_20260731.zip to Signal_or_Redundancy_R_FULL_LOGHAR_FIXED_20160801_20260731.zip
Using R ZIP: /content/Signal_or_Redundancy_R_FULL_LOGHAR_FIXED_20160801_20260731.zip
Frozen panel rows: 71040
Stocks: 29
Date range: 2016-08-01 to 2026-07-31

================ LIGHTGBM TUNING STARTED ================
Selected LightGBM configurations:
   horizon candidate  mean_validation_QLIKE  fixed_test_rounds
0        5      LGB5               0.413084                 36
1       10      LGB5               0.279424                 38
2       20      LGB5               0.197852                 40

================ LIGHTGBM LOCKED TEST STARTED ================
LightGBM locked forecast rows: 282312

================ BUILDING LSTM SEQUENCES ================

================ LSTM TUNING STARTED ================
Selected LSTM configurations:
   horizon candidate  mean_validation_QLIKE  fixed_test_epochs
0        5     LSTM1               0.434558         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>